# Test avec CatBoostRegressor

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

In [15]:
df = pd.read_csv("../data/gold.csv")

In [16]:
# column_titles = df.columns.tolist()
# column_titles

In [ ]:
X = df.drop(["prix_median"], axis=1)
y = df['prix_median']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, shuffle=True, random_state=42)

categorial_features = ["proximité_autoroute"]

numerical_features = ['tx_crim', 'tx_residence','tx_commerce',
       'tx_nitriq', 'nb_piece', 'tx_ancienneté_parc_immo', 'distance_centre_emploi',
        'indice_impot_foncier',  'ratio_eleve_enseignant', 'tx_status_sociaux_eco_inf']

categorical_transformer = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
numerical_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorial_features),
        ('num', numerical_transformer, numerical_features)
    ],
    remainder="passthrough" 
)

# Créer un objet CatBoostRegressor
# model = CatBoostRegressor(random_state=42, verbose=False)
# model = CatBoostRegressor(depth=4, learning_rate=0.19293408926881367, l2_leaf_reg=1.9445149121337306, random_state=42, verbose=False)
# model = CatBoostRegressor(iterations = 700, learning_rate = 0.0693060439562983, depth = 6, loss_function = 'RMSE', subsample = 0.30000000000000004, l2_leaf_reg = 0.9500743944149637, random_strength = 0.10292853111379426, random_state=42, verbose=False)
model = CatBoostRegressor(iterations = 900, learning_rate = 0.0677148890414124, depth = 9, loss_function = 'RMSE', subsample = 0.5, l2_leaf_reg = 9.760330771250379, random_strength = 4.695597681100087, random_state=42, verbose=False)


# Créer un pipeline avec le préprocesseur et le modèle CatBoostRegressor
pipe = Pipeline([
     ('preprocessor', preprocessor),
     ('model', model)
])

# Entraîner le pipeline sur les données d'entraînement
pipe.fit(X_train, y_train)
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

print("Mean squared error : ")
print("TRAIN :",mean_squared_error(y_train, y_pred_train))
print("TEST :",mean_squared_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("Mean absolute error : ")
print("TRAIN :",mean_absolute_error(y_train, y_pred_train))
print("TEST :",mean_absolute_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("R2 score : ")    
print("TRAIN :",r2_score(y_train, y_pred_train))
print("TEST :",r2_score(y_test, y_pred_test))

### Optuna (recherche des hyperparametres)

In [ ]:
import optuna

In [ ]:
def objective(trial):
    # Définition des hyperparamètres à optimiser
    iterations = trial.suggest_int('iterations', 100, 1000, step=100)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
    depth = trial.suggest_int('depth', 3, 10)
    loss_function = trial.suggest_categorical('loss_function', ['RMSE', 'MAE'])
    subsample = trial.suggest_float('subsample', 0.1, 1.0, step=0.1)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 0.1, 10.0)
    random_strength = trial.suggest_float('random_strength', 0.1, 10.0)
    
    # Création du modèle avec les hyperparamètres suggérés
    model = CatBoostRegressor(iterations=iterations,
                              learning_rate=learning_rate,
                              depth=depth,
                              loss_function=loss_function,
                              subsample=subsample,
                              l2_leaf_reg=l2_leaf_reg,
                              random_strength=random_strength,
                              random_state=42,
                              verbose=False,
                              allow_writing_files=False)
    
    # Même pipeline (préprocesseur + modèle) que pour l'évaluation finale
    pipe = Pipeline([
         ('preprocessor', preprocessor),
         ('model', model)
    ])
    
    # Validation croisée sur le jeu d'entraînement uniquement :
    # le jeu de test reste réservé à l'évaluation finale
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1).mean()
    
    return r2

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

In [ ]:
best_params = study.best_params
best_value = study.best_value

In [ ]:
print(best_params)
print(best_value)

### Évaluation finale sur le jeu de test

In [ ]:
# Pipeline avec les meilleurs hyperparamètres trouvés par Optuna,
# entraîné sur le jeu d'entraînement puis évalué une seule fois sur le jeu de test
best_pipe = Pipeline([
     ('preprocessor', preprocessor),
     ('model', CatBoostRegressor(**best_params, random_state=42, verbose=False))
])
best_pipe.fit(X_train, y_train)
y_pred_test = best_pipe.predict(X_test)

print("TEST MSE :", mean_squared_error(y_test, y_pred_test))
print("TEST MAE :", mean_absolute_error(y_test, y_pred_test))
print("TEST R2 :", r2_score(y_test, y_pred_test))